In [0]:
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA sliver")

In [0]:
customer_bronze_df = spark.table("retail_lakehouse.bronze.customers")
display(customer_bronze_df)


In [0]:
customer_bronze_df.printSchema()

In [0]:
from pyspark.sql import functions as F

# dups check

dups_check = (
    customer_bronze_df
    .groupBy('customer_id')
    .count()
    .filter(F.col('count') > 1)
)

display(dups_check)

display(
    customer_bronze_df
    .filter(F.col('customer_id') == 'C0040')
)

In [0]:
# drop the duplicates

customer_dedup_df = customer_bronze_df.dropDuplicates(["customer_id"])

customer_dedup_df.count()

In [0]:
# check after remove the dups
from pyspark.sql import functions as F

(
    customer_dedup_df
    .groupBy('customer_id')
    .count()
    .filter(F.col('count') > 1)
    .show()

)


In [0]:
# Null checks on customer_dedup_df 

nulls_df = (
    customer_dedup_df
    .select(
        *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in customer_dedup_df.columns]
    )
)

nulls_df.display()


In [0]:
# standardize the customer state
state_map_df = (
    spark.table("retail_lakehouse.reference.state_mapping")
    .select(
        F.upper(F.trim(F.col("source_state"))).alias("state_key"),
        F.col("standard_state")
    )
)

display(state_map_df)

In [0]:
# standardize the customer state

customer_state_map = (
    customer_dedup_df
    .withColumn(
        "state_key",
        F.upper(F.trim(F.col("state")))
    )
    .join(
        state_map_df,
        on="state_key",
        how="left"
    )
)

In [0]:
# run the cleand DF
customer_clean_df = (
    customer_state_map
    .withColumn(
        "state_clean",
        F.coalesce(
            F.col("standard_state"),
            F.col("state_key")
        )
    )
)

In [0]:
display(
    customer_clean_df.select(
        "customer_id",
        "state",
        "standard_state",
        "state_clean"
    )
)



In [0]:
customer_clean_df.select("state_clean").distinct().display()

In [0]:
# convert the created_at timestamp it was string in bronze table

customer_clean_df = (
    customer_state_map
    .withColumn(
        "created_at",
        F.to_timestamp(
            F.col("created_at"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    .withColumn(
        "state_clean",
        F.coalesce(
            F.col("standard_state"),
            F.col("state_key")
        )
    )
)

display(customer_clean_df)

In [0]:
# write into sliver table

customer_sliver_df = (
    customer_clean_df
    .drop("state_clean", "state_key", "state")
    .withColumnRenamed("standard_state", "state")
)




In [0]:
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA sliver")

In [0]:
# cretae slive table
customer_clean_df = (
    customer_sliver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail_lakehouse.sliver.customers")
)
